In [4]:
import torchvision
import torch

# 指定 transform 将把原本是 PIL Image 格式的数据转为 PyTorch 需要的 Tensor 格式
train_data = torchvision.datasets.CIFAR10(root="./dataset", train=True, transform=torchvision.transforms.ToTensor(), download=True)
test_data = torchvision.datasets.CIFAR10(root="./dataset", train=False, transform=torchvision.transforms.ToTensor(), download=True)

# length 长度
train_data_length = len(train_data)
test_data_length = len(test_data)

# 如果train_data_length = 10, 则训练数据集的长度为10
print("训练数据集的长度为：{}".format(train_data_length))
print("测试数据集的长度为：{}".format(test_data_length))
# 也可以使用f-string的方式来打印长度
# print(f"训练数据集的长度: {train_data_length}")
# print(f"测试数据集的长度: {test_data_length}")

# ！！！！定义设备，优先使用GPU，如果没有GPU则使用CPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


# 利用 DataLoader 来加载数据集
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False)


# from P27_model import *
# 创建神经网络模型
# 搭建神经网络
import torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(1024, 64),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.model(x)
        return x
    
net = Net()

net = net.to(device)  # ！！！ 用to(device)将模型移动到 GPU 上进行训练

# 损失函数
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()
loss_fn = loss_fn.to(device)  # 将损失函数移动到 GPU 上进行计算

# 优化器
learning_rate =   0.005  #推荐学习率一般   
optimizer = torch.optim.SGD(net.parameters(), lr=learning_rate)
# 设置训练参数
# 记录训练的次数
total_train_step = 0
# 记录测试的次数
total_test_step = 0
# 训练的轮数
epoch = 50

# 添加tensorboard
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("logs")

import time
start_time = time.time()  # 记录训练开始的时间

import os
# 确保保存模型的文件夹存在，如果不存在则自动创建
os.makedirs("models", exist_ok=True)

for i in range(epoch):
    print("第 {} 轮训练开始".format(i+1))
    
    # 按照之前的优化建议，将 net.train() 提到了每个 epoch 循环一开始
    net.train()
    
    # 训练步骤开始
    for data in train_dataloader:
        # data 是一个列表，包含了训练数据和对应的标签
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        # 将输入数据传入神经网络模型，得到输出结果
        outputs = net(images)
        # 计算损失函数的值
        loss = loss_fn(outputs, labels)
        # 优化器清零
        optimizer.zero_grad()   
        # 反向传播，计算梯度
        loss.backward()
        # 更新参数
        optimizer.step()
        total_train_step += 1
        if total_train_step % 100 == 0:
            end_time = time.time()  # 记录当前时间
            
            print("训练次数: {}, Loss: {}".format(total_train_step, loss.item()))
            print("每 100 次训练的时间: {} 秒".format(end_time - start_time))
            
            writer.add_scalar("train_loss", loss.item(), total_train_step)
            # 记录训练的次数和损失值到 TensorBoard 中，方便后续可视化分析


    # 测试步骤开始
    with torch.no_grad():   # 在测试阶段，我们不需要计算梯度，因此使用 torch.no_grad() 来禁用梯度计算
        net.eval()  # 将模型设置为评估模式
        total_test_loss = 0
        total_accuracy = 0
        for data in test_dataloader:
            images, labels = data
            images = images.to(device)
            labels = labels.to(device)
            outputs = net(images)
            loss = loss_fn(outputs, labels)
            total_test_loss += loss.item()
            # 计算准确率
            accuracy = (outputs.argmax(dim=1) == labels).sum().item()  # 计算当前批次中预测正确的数量
            total_accuracy += accuracy  # 累加预测正确的数量
            # _, predicted = torch.max(outputs.data, 1)  # 获取输出结果中概率最大的类别索引
            # total_accuracy += (predicted == labels).sum().item()  # 计算预测正确的数量
    print("整体测试集上的Loss: {}".format(total_test_loss))
    print("整体测试集上的准确率: {}%".format(total_accuracy / test_data_length * 100))
    writer.add_scalar("test_loss", total_test_loss, total_test_step)
    writer.add_scalar("test_accuracy", total_accuracy / test_data_length, total_test_step)
    total_test_step += 1

    # 每个 epoch 结束后保存一次模型，路径指向 models 文件夹
    torch.save(net, "./models/net_{}.pth".format(i+1))
    # torch.save(net.state_dict(), "./models/net_{}_state_dict.pth".format(i+1))    仅保存模型参数
    print("模型已保存至 ./models/net_{}.pth".format(i+1))

writer.close()

Files already downloaded and verified
Files already downloaded and verified
训练数据集的长度为：50000
测试数据集的长度为：10000
第 1 轮训练开始
训练次数: 100, Loss: 2.3121795654296875
每 100 次训练的时间: 0.3390491008758545 秒
训练次数: 200, Loss: 2.289949893951416
每 100 次训练的时间: 0.6541473865509033 秒
训练次数: 300, Loss: 2.286644697189331
每 100 次训练的时间: 0.9611668586730957 秒
训练次数: 400, Loss: 2.2943193912506104
每 100 次训练的时间: 1.2620751857757568 秒
训练次数: 500, Loss: 2.290864944458008
每 100 次训练的时间: 1.562084674835205 秒
训练次数: 600, Loss: 2.274533271789551
每 100 次训练的时间: 1.8636715412139893 秒
训练次数: 700, Loss: 2.2204642295837402
每 100 次训练的时间: 2.1652281284332275 秒
整体测试集上的Loss: 350.84277272224426
整体测试集上的准确率: 19.650000000000002%
模型已保存至 ./models/net_1.pth
第 2 轮训练开始
训练次数: 800, Loss: 2.207134962081909
每 100 次训练的时间: 2.884432792663574 秒
训练次数: 900, Loss: 2.188217878341675
每 100 次训练的时间: 3.2140355110168457 秒
训练次数: 1000, Loss: 2.057128667831421
每 100 次训练的时间: 3.526737689971924 秒
训练次数: 1100, Loss: 2.0869510173797607
每 100 次训练的时间: 3.8474972248077393 秒
训练次数: 120